# RAG POC

## Stage 2: Ingest

Goal: turn raw MDX files into clean `Document`s, meaning text plus metadata.
Everything later depends on this stage. If the noise isn't removed here, it gets embedded and retrieved.

In [1]:
import re
from pathlib import Path
from langchain_core.documents import Document

DOCS = Path("data/langchain-docs/src/oss")

# ponytail: hand-picked subset (concepts + RAG/retrieval + core LangChain). Add pages later to see retrieval degrade.
SUBSET = [
    "langchain/overview", "langchain/philosophy", "langchain/quickstart", "langchain/component-architecture",
    "langchain/agents", "langchain/models", "langchain/messages", "langchain/tools",
    "langchain/structured-output", "langchain/short-term-memory", "langchain/long-term-memory",
    "langchain/streaming", "langchain/context-engineering", "langchain/guardrails",
    "langchain/human-in-the-loop", "langchain/knowledge-base", "langchain/retrieval",
    "langchain/middleware/overview", "langchain/middleware/built-in",
    "concepts/context", "concepts/memory", "concepts/providers-and-models", "concepts/products",
    "deepagents/retrieval", "deepagents/rag", "langgraph/overview", "langgraph/agentic-rag",
]

In [2]:
FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n", re.S)

def load(rel: str) -> Document:
    raw = (DOCS / f"{rel}.mdx").read_text()
    fm = FRONTMATTER.match(raw)
    title = re.search(r"^title:\s*(.+)$", fm.group(1), re.M) if fm else None
    body = raw[fm.end():] if fm else raw
    return Document(page_content=body, metadata={"source": rel, "title": title.group(1).strip('"') if title else rel})

raw_docs = [load(r) for r in SUBSET]
print(len(raw_docs), "docs |", sum(len(d.page_content) for d in raw_docs), "chars")

27 docs | 797759 chars


### Cleaning MDX

The noise found in these files:
- `import X from '/snippets/...'`: content pulled in from other files that we **don't** load
- `:::python ... :::` and `:::js ... :::`: the **same content written twice**, once per language
- JSX components: `<Note>`, `<Tip>`, `<CodeGroup>`, `<Tabs>`, `<ParamField path="...">`, ...
- `@[\`Document\`]`: the docs' own cross-reference syntax

In [3]:
def clean_mdx(text: str) -> str:
    text = re.sub(r"^[ \t]*:::js\n.*?^[ \t]*:::[ \t]*\n?", "", text, flags=re.S | re.M)        # drop JS duplicate content
    text = re.sub(r"^[ \t]*:::(?:python)?[ \t]*\n?", "", text, flags=re.M)             # unwrap Python blocks
    text = re.sub(r"^import .*$\n?", "", text, flags=re.M)                     # snippet imports (content not loaded)
    text = re.sub(r'<ParamField\s+[^>]*?path="([^"]+)"[^>]*>', r"Parameter `\1`:", text)  # keep param name
    text = re.sub(r"</?[A-Z][A-Za-z]*(?:\s[^>]*)?/?>", "", text)               # strip JSX tags, keep inner text
    text = re.sub(r"@\[(`?[^\]]+`?)\]", r"\1", text)                          # @[`Document`] -> `Document`
    return re.sub(r"\n{3,}", "\n\n", text).strip()                            # collapse blank lines left behind

In [4]:
docs = [Document(page_content=clean_mdx(d.page_content), metadata=d.metadata) for d in raw_docs]

before, after = sum(len(d.page_content) for d in raw_docs), sum(len(d.page_content) for d in docs)
print(f"chars: {before} -> {after} ({after/before:.0%} kept)")
for d in docs:
    if len(d.page_content.strip()) < 300:
        print("STUB?", d.metadata["source"], repr(d.page_content[:80]))

# Leftover noise check: should all be 0 or near it after cleaning
for name, pat in {"import": r"^import ", ":::js": r":::js", "JSX tag": r"</?[A-Z][A-Za-z]*", "@[ref]": r"@\["}.items():
    print(f"{name:8}", sum(len(re.findall(pat, d.page_content, re.M)) for d in docs))

chars: 797759 -> 502569 (63% kept)
STUB? langchain/retrieval ''
import   0
:::js    0
JSX tag  0
@[ref]   0


In [5]:
# Inspect: read a cleaned page the way the embedder will see it
print(docs[SUBSET.index("langchain/knowledge-base")].page_content[:2500])

## Overview

Build a semantic search engine over a PDF with LangChain [embeddings](/oss/integrations/embeddings) and [vector stores](/oss/integrations/vectorstores). Use it to retrieve passages similar to a query, then plug the retriever into [retrieval-augmented generation (RAG)](/oss/deepagents/retrieval) or other LLM workflows.

This tutorial covers:

1. Create `Document` objects from a PDF.
2. Generate embeddings.
3. Load and split a PDF.
4. Index chunks in a vector store and query by similarity.
5. Wrap the store as a retriever.

The guide also includes a minimal RAG implementation on top of the search engine.

### Concepts

This tutorial focuses on text retrieval and covers the following concepts:

- `Document`
- [Text splitters](/oss/integrations/splitters)
- [Embeddings](/oss/integrations/embeddings)
- [Vector stores](/oss/integrations/vectorstores) and [retrievers](/oss/integrations/retrievers)

## Setup

### Install dependencies

This tutorial reads a PDF using the `pypdf` pa

## Stage 3: Chunk

Why chunk at all: one vector per *page* averages many topics into one blurry point, so it matches everything a little and nothing well.
One vector per *chunk* is sharp, but a chunk cut in the wrong place loses its meaning (half a code block, a sentence without its heading).

Baseline first (naive), then measure what went wrong.

In [6]:
docs = [d for d in docs if len(d.page_content) >= 300]   # drop stub pages
print(len(docs), "docs")

26 docs


In [7]:
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
tok = AutoTokenizer.from_pretrained(os.environ["EMBED_MODEL"])  # the embedder counts tokens, not characters

def diagnose(chunks, name):
    lens = [len(c.page_content) for c in chunks]
    toks = [len(tok(c.page_content)["input_ids"]) for c in chunks]
    pct = lambda xs: [sorted(xs)[int(p * (len(xs) - 1))] for p in (0, .5, .95, 1)]
    cut_code = sum(c.page_content.count("```") % 2 for c in chunks)                   # odd ``` count = code block cut in half
    mid_start = sum(not re.match(r"[#A-Z`\-\d*\[|>]", c.page_content.lstrip()) for c in chunks)  # starts mid-sentence
    tiny = sum(l < 150 for l in lens)
    print(f"{name}: {len(chunks)} chunks | chars min/p50/p95/max {pct(lens)} | tokens {pct(toks)}")
    print(f"  cut code blocks: {cut_code} ({cut_code/len(chunks):.0%}) | mid-sentence starts: {mid_start} ({mid_start/len(chunks):.0%}) | tiny(<150): {tiny}")

In [8]:
baseline = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, add_start_index=True).split_documents(docs)
diagnose(baseline, "baseline")

baseline: 813 chunks | chars min/p50/p95/max [3, 679, 786, 800] | tokens [3, 162, 225, 343]
  cut code blocks: 270 (33%) | mid-sentence starts: 206 (25%) | tiny(<150): 14


In [9]:
# Inspect the failures: a chunk with half a code block, and a tiny one
bad_code = next(c for c in baseline if c.page_content.count("```") % 2)
tiny = min(baseline, key=lambda c: len(c.page_content))
for label, c in [("CUT CODE", bad_code), ("TINY", tiny)]:
    print(f"--- {label} | {c.metadata['source']} @ {c.metadata['start_index']}\n{c.page_content}\n")

--- CUT CODE | langchain/overview @ 1334
This example demonstrates how to create a simple LangChain agent with a custom tool:

    ```python OpenAI
    # pip install -qU langchain "langchain[openai]"
    from langchain.agents import create_agent

    def get_weather(city: str) -> str:
        """Get weather for a given city."""
        return f"It's always sunny in {city}!"

    agent = create_agent(
        model="openai:gpt-5.5",
        tools=[get_weather],
        system_prompt="You are a helpful assistant",
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
    )
    print(result["messages"][-1].content_blocks)
    ```
    ```python Google Gemini
    # pip install -qU langchain "langchain[google-genai]"
    from langchain.agents import create_agent

--- TINY | langchain/streaming @ 11546
```



### Smarter separators

`RecursiveCharacterTextSplitter` tries separators **in order**: it splits on the first one, and only splits pieces that are still too big using the next one.
Default `["\n\n", "\n", " ", ""]` knows nothing about markdown. Your list decides where cuts are *preferred*.

In [10]:
from langchain_text_splitters import Language

# Headings first (any level), then fences, horizontal rules, paragraphs, lines, words.
# They are regexes, so is_separator_regex=True matters: without it they never match.
MD_SEPARATORS = RecursiveCharacterTextSplitter.get_separators_for_language(Language.MARKDOWN)
print(MD_SEPARATORS)

mine = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, separators=MD_SEPARATORS,
                                      is_separator_regex=True, add_start_index=True).split_documents(docs)
diagnose(baseline, "baseline")
diagnose(mine, "markdown")

['\n#{1,6} ', '```\n', '\n\\*\\*\\*+\n', '\n---+\n', '\n___+\n', '\n\n', '\n', ' ', '']


baseline: 813 chunks | chars min/p50/p95/max [3, 679, 786, 800] | tokens [3, 162, 225, 343]
  cut code blocks: 270 (33%) | mid-sentence starts: 206 (25%) | tiny(<150): 14


markdown: 1003 chunks | chars min/p50/p95/max [3, 553, 778, 798] | tokens [3, 133, 215, 335]
  cut code blocks: 346 (34%) | mid-sentence starts: 138 (14%) | tiny(<150): 90
